In [1]:
!pip install folium

In [2]:
import pandas as pd
import folium
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, cos, sin, asin, sqrt

In [3]:
df_transactions = pd.read_csv('merged_transactions.csv')
df_transactions.head(6)

,NO_TOKEN_CB,CD_TICKET_UNIQUE,DT_VENTE,TS_VENTE,ID_MAG_TIERS,ID_ARTICLE,MT_TTC_NET,TX_TVA,QT_UVC,QT_POIDS,...,LB_DESIGNATION_LIEU_DE_VENTE,LB_ADRESSE,CD_POSTAL,LB_VILLE,LATITUDE,LONGITUDE,DT_OUVERTURE,TOTAL_QTY,QTY_PROMO_NAT,QTY_PROMO_STORE
0,A33FB7919EEAB875A8AF4493286211B29F49B0746939EF...,4949207601061614291,2025-12-05,2025-12-05 08:46:00,79,100000000008209,1.09,0.055,0.55,0.550,...,Vaulx en Velin,2 RUE JEAN ET JOSEPHINE PEYRI,69120,VAULX EN VELIN,45.785683,4.926692,2007-05-16,1.100,0.0,0.0
1,A33FB7919EEAB875A8AF4493286211B29F49B0746939EF...,4949207601061614291,2025-12-05,2025-12-05 08:46:00,79,100000000009130,0.50,0.055,0.20,0.200,...,Vaulx en Velin,2 RUE JEAN ET JOSEPHINE PEYRI,69120,VAULX EN VELIN,45.785683,4.926692,2007-05-16,0.400,0.0,0.0
2,2755B58F8A00BA3FA60E9E69A1951194F048FDF5FB0C2C...,4945949002201926787,2025-12-01,2025-12-01 14:15:00,1296,100000000009179,0.99,0.055,1.00,0.000,...,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680090,3.138073,2018-02-14,1.000,0.0,0.0
3,2755B58F8A00BA3FA60E9E69A1951194F048FDF5FB0C2C...,4945949002201926787,2025-12-01,2025-12-01 14:15:00,1296,100000000005706,1.56,0.055,0.19,0.195,...,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680090,3.138073,2018-02-14,0.385,0.0,0.0
4,7B6704A84131A41FF8F2CB121FD2517FE65F52B714B4C4...,4964269402201934474,2025-12-22,2025-12-22 19:09:00,1297,80000000000C1HB,5.39,0.055,1.00,0.110,...,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680090,3.138073,2018-02-14,1.110,0.0,0.0
5,7B6704A84131A41FF8F2CB121FD2517FE65F52B714B4C4...,4964269402201934474,2025-12-22,2025-12-22 19:09:00,1297,80000000000C2OR,4.34,0.055,1.00,0.217,...,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680090,3.138073,2018-02-14,1.217,0.0,0.0


In [4]:
# store info list
unique_stores = df_transactions[[
    'ID_LIEU_DE_VENTE',              
    'LB_DESIGNATION_LIEU_DE_VENTE',
    'LB_VILLE',                      
    'LATITUDE',                    
    'LONGITUDE'                     
]].drop_duplicates().sort_values('ID_LIEU_DE_VENTE')


unique_stores.reset_index(drop=True, inplace=True)

print(f"Number of stores: {len(unique_stores)}。")
display(unique_stores)

invalid_coords = unique_stores[
    (unique_stores['LATITUDE'] == 0) | 
    (unique_stores['LONGITUDE'] == 0)
]

if len(invalid_coords) > 0:
    print(f"\n Warning：have {len(invalid_coords)} coordinate seem invalid (equal 0):")
    display(invalid_coords)
else:
    print("\n all coordinates are valid (None 0)。")

Number of stores: 11。


,ID_LIEU_DE_VENTE,LB_DESIGNATION_LIEU_DE_VENTE,LB_VILLE,LATITUDE,LONGITUDE
0,106,Vaulx en Velin,VAULX EN VELIN,45.785683,4.926692
1,163,Perpignan,PERPIGNAN,42.671413,2.888161
2,180,Meyzieu,MEYZIEU,45.772190,4.987101
3,195,Hautepierre,STRASBOURG,48.587494,7.700943
4,220,Wasquehal,Wasquehal,50.680090,3.138073
5,277,Polygone,PERPIGNAN,42.738544,2.890626
6,286,Fleury sur Orne,Fleury Sur Orne,49.141180,-0.372080
7,414,Mions,Mions,45.680557,4.956442
8,532,Decines,DECINES-CHARPIEU,45.746082,4.934131
9,534,Bron,BRON,45.727703,4.915211



 all coordinates are valid (None 0)。


In [14]:
# map stores location
center_lat = unique_stores['LATITUDE'].mean()
center_lon = unique_stores['LONGITUDE'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

for idx, row in unique_stores.iterrows():
    folium.Marker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        popup=f"{row['LB_DESIGNATION_LIEU_DE_VENTE']} ({row['LB_VILLE']})",
        tooltip=f"Store ID: {row['ID_LIEU_DE_VENTE']}",
        icon=folium.Icon(color='blue', icon='shopping-cart')
    ).add_to(m)

print("Store location map：")
m

Store location map：


In [15]:
print("Analyze Adjacent Stores")

# 1. Haversine Formula
def calculate_distance(row):
    coord_a = store_coords.get(row['ID_LIEU_DE_VENTE_A'])
    coord_b = store_coords.get(row['ID_LIEU_DE_VENTE_B'])
    
    if not coord_a or not coord_b:
        return None

    lon1, lat1, lon2, lat2 = map(radians, [
        coord_a['LONGITUDE'], coord_a['LATITUDE'], 
        coord_b['LONGITUDE'], coord_b['LATITUDE']
    ])

    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r

Analyze Adjacent Stores


In [17]:
# 2. calculate distance between each store

# df_adjacency = links.copy()

print("Calculating distances between store pairs")
df_adjacency['distance_km'] = df_adjacency.apply(calculate_distance, axis=1)

df_adjacency['Store_Name_A'] = df_adjacency['ID_LIEU_DE_VENTE_A'].map(store_names)
df_adjacency['Store_Name_B'] = df_adjacency['ID_LIEU_DE_VENTE_B'].map(store_names)

NameError: name 'links' is not defined

In [ ]:
# 3. filter adjacent stores
adjacency_threshold_km = 20

adjacent_pairs = df_adjacency[
    (df_adjacency['distance_km'] <= adjacency_threshold_km) & 
    (df_adjacency['shared_customers'] > 0)
].sort_values(by='shared_customers', ascending=False)

print(f"\n--- Top 10 Adjacent Store Pairs (Distance < {adjacency_threshold_km}km) with Highest Shared Traffic ---")
display_cols = ['Store_Name_A', 'Store_Name_B', 'distance_km', 'shared_customers']
display(adjacent_pairs[display_cols].head(10))

In [ ]:
# 4. correlation analytic
plt.figure(figsize=(10, 6))

# distance (set life circle in 100km)
data_filtered = df_adjacency[df_adjacency['distance_km'] < 100]

# calculate correlation
corr = data_filtered[['distance_km', 'shared_customers']].corr().iloc[0,1]

sns.regplot(
    data=data_filtered, 
    x='distance_km', 
    y='shared_customers', 
    scatter_kws={'alpha': 0.6}, 
    line_kws={'color': 'red', 'label': f'Linear Trend (Corr: {corr:.2f})'}
)


plt.title(f'Relationship between Distance and Shared Customers (Pairs < 100km)')
plt.xlabel('Distance between Stores (km)')
plt.ylabel('Number of Shared Customers')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend() 
plt.show()

print(f"Correlation between Distance and Shared Customers: {corr:.2f}")

In [ ]:
print(f"Drawing Map for {len(adjacent_pairs)} adjacent store pairs...")

# 1. Prepare Map Data
relevant_store_ids = set(adjacent_pairs['ID_LIEU_DE_VENTE_A']).union(set(adjacent_pairs['ID_LIEU_DE_VENTE_B']))

filtered_stores_list = []
for store_id in relevant_store_ids:
    if store_id in store_coords:
        store_data = store_coords[store_id]
        store_data['ID'] = store_id
        store_data['NAME'] = store_names.get(store_id, f"Store {store_id}")
        filtered_stores_list.append(store_data)

# Bounds
lats = [s['LATITUDE'] for s in filtered_stores_list]
lons = [s['LONGITUDE'] for s in filtered_stores_list]

if lats and lons:
    min_lat, max_lat = min(lats), max(lats)
    min_lon, max_lon = min(lons), max(lons)
else:
    min_lat, max_lat = 42.0, 51.0
    min_lon, max_lon = -5.0, 8.0


In [ ]:
# 2. Initialize Map
m = folium.Map(tiles='cartodbpositron') 
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

# 3. Draw Connection Lines
if not adjacent_pairs.empty:
    max_weight = adjacent_pairs['shared_customers'].max()

    for _, row in adjacent_pairs.iterrows():
        loc_a = [store_coords[row['ID_LIEU_DE_VENTE_A']]['LATITUDE'], store_coords[row['ID_LIEU_DE_VENTE_A']]['LONGITUDE']]
        loc_b = [store_coords[row['ID_LIEU_DE_VENTE_B']]['LATITUDE'], store_coords[row['ID_LIEU_DE_VENTE_B']]['LONGITUDE']]
        
        folium.PolyLine(
            locations=[loc_a, loc_b],
            weight=(row['shared_customers'] / max_weight) * 5 + 1, 
            color='#E74C3C',
            opacity=0.7,
            tooltip=f"{row['Store_Name_A']} <-> {row['Store_Name_B']}<br>Shared: {row['shared_customers']}<br>Dist: {row['distance_km']:.1f}km"
        ).add_to(m)

# 4. Draw Store Markers
for store in filtered_stores_list:
    folium.CircleMarker(
        location=[store['LATITUDE'], store['LONGITUDE']],
        radius=6,
        color='#2980B9', 
        fill=True,
        fill_color='#2980B9',
        popup=f"{store['NAME']} (ID: {store['ID']})"
    ).add_to(m)

# 5. show map
m